## 환경설정

In [ ]:
!pip install adjustText -qq
from adjustText import adjust_text

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import koreanize_matplotlib
from sqlalchemy import create_engine

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

engine = create_engine(os.environ["DB_URL"])
conn = engine.connect()

In [ ]:
DATE_FMT = "%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f"

RETENTION_START_HOUR = 24
RETENTION_WINDOW_DAY = 7

In [ ]:
def _normalize_sql(sql: str) -> str:

    return sql.replace("%%", "%")


def run_query(query: str, name: str | None = None) -> pd.DataFrame:
    result = conn.exec_driver_sql(_normalize_sql(query))
    rows = result.fetchall()
    df = pd.DataFrame(rows, columns=result.keys())
    if name:
        print(f"[{name}] rows={len(df):,}, cols={len(df.columns):,}")
    return df


def execute_many(sql: str) -> None:
    statements = [stmt.strip() for stmt in sql.split(";") if stmt.strip()]
    for stmt in statements:
        conn.exec_driver_sql(_normalize_sql(stmt))
    try:
        conn.commit()
    except Exception:
        pass
    print(f"Executed {len(statements):,} statements.")

## 데이터 전처리

### VIEW 생성 및 결측값 확인

- 이 과정에서 user_id 결측 + event_time 변환 실패 행 제거

In [ ]:
create_views_sql = """
DROP VIEW IF EXISTS v_events_signup;
CREATE VIEW v_events_signup AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time
FROM events_signup
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_content_start;
CREATE VIEW v_events_content_start AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_start
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_lesson_view;
CREATE VIEW v_events_lesson_view AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_view
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_lesson_complete;
CREATE VIEW v_events_lesson_complete AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_complete
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_content_end;
CREATE VIEW v_events_content_end AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_end
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_related_question_click;
CREATE VIEW v_events_related_question_click AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id`  AS content_id,
    `lesson_id`   AS lesson_id
FROM events_related_question_click
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;
"""

execute_many(create_views_sql)

### VIEW 생성 여부 검증

In [ ]:
run_query("""
SELECT 'v_events_signup'        AS view_name, COUNT(*) AS row_cnt FROM v_events_signup
UNION ALL SELECT 'v_events_content_start',           COUNT(*) FROM v_events_content_start
UNION ALL SELECT 'v_events_lesson_view',       COUNT(*) FROM v_events_lesson_view
UNION ALL SELECT 'v_events_lesson_complete',         COUNT(*) FROM v_events_lesson_complete
UNION ALL SELECT 'v_events_content_end',             COUNT(*) FROM v_events_content_end
UNION ALL SELECT 'v_events_related_question_click',  COUNT(*) FROM v_events_related_question_click;
""", "view_check")

### 결측값 확인

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

null_check_df = run_query(f"""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END) AS user_id_null,
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END) AS event_time_null
FROM events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_related_question_click;
""", "null_check")

null_check_df

### 중복값 확인

In [ ]:
duplicate_check_df = run_query("""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    COUNT(DISTINCT user_id, event_time) AS unique_cnt,
    COUNT(*) - COUNT(DISTINCT user_id, event_time) AS duplicated_cnt
FROM v_events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_related_question_click;
""", "duplicate_check")

duplicate_check_df

### 이상치 1 : 시간 범위

In [ ]:
time_range_df = run_query("""
SELECT 'v_events_signup' AS view_name,
    MIN(event_time) AS min_t, MAX(event_time) AS max_t,
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END) AS future_cnt
FROM v_events_signup
UNION ALL
SELECT 'v_events_content_start', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_start
UNION ALL
SELECT 'v_events_lesson_view', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_view
UNION ALL
SELECT 'v_events_lesson_complete', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_complete
UNION ALL
SELECT 'v_events_content_end', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_end
UNION ALL
SELECT 'v_events_related_question_click', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_related_question_click;
""", "time_range")

time_range_df

### 이상치 2 : 가입 전 활동 (정합성)

In [ ]:
before_signup_df = run_query("""
WITH signup AS (
    SELECT user_id, MIN(event_time) AS signup_time
    FROM v_events_signup GROUP BY user_id
)
SELECT 'v_events_content_start' AS view_name,
    COUNT(*) AS before_signup_rows
FROM v_events_content_start sc
JOIN signup s ON sc.user_id = s.user_id
WHERE sc.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_view', COUNT(*)
FROM v_events_lesson_view l
JOIN signup s ON l.user_id = s.user_id
WHERE l.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_complete', COUNT(*)
FROM v_events_lesson_complete cl
JOIN signup s ON cl.user_id = s.user_id
WHERE cl.event_time < s.signup_time
UNION ALL
SELECT 'v_events_content_end', COUNT(*)
FROM v_events_content_end ec
JOIN signup s ON ec.user_id = s.user_id
WHERE ec.event_time < s.signup_time
UNION ALL
SELECT 'v_events_related_question_click', COUNT(*)
FROM v_events_related_question_click cq
JOIN signup s ON cq.user_id = s.user_id
WHERE cq.event_time < s.signup_time;
""", "before_signup")

before_signup_df

### 이상치 3 : 봇 의심 (한 유저가 너무 많은 이벤트)

- 유저별 일평균 활동량 분포 살펴보기

In [ ]:
user_stats_query = """
    SELECT
        user_id,
        COUNT(*) AS event_cnt,
        TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)) AS active_days,
        COUNT(*) / GREATEST(TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)), 1) AS daily_avg
    FROM v_events_lesson_view
    GROUP BY user_id
"""
df_user_stats = run_query(user_stats_query)

In [ ]:
print("전체 유저 수:", len(df_user_stats))
print("\n=== daily_avg 분포 ===")
print(df_user_stats['daily_avg'].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]))

print("\n=== 임계값별 봇으로 분류되는 유저 수 ===")
for threshold in [20, 30, 50, 100, 200, 500, 974]:
    bot_count = (df_user_stats['daily_avg'] >= threshold).sum()
    pct = bot_count / len(df_user_stats) * 100
    print(f"  ≥ {threshold:>4}/일 : {bot_count:>6}명 ({pct:.3f}%)")

- 전체 11.4만 명

- 974/일 이상: 8명
- 500/일 이상: 93명
- 200/일 이상: 441명

In [ ]:
top_suspects = df_user_stats[df_user_stats['daily_avg'] >= 500].sort_values('daily_avg', ascending=False)
print(f"500/일 이상 유저: {len(top_suspects)}명")
print("\n=== 상위 20명 ===")
print(top_suspects[['user_id', 'event_cnt', 'active_days', 'daily_avg']].head(20))

print("\n=== 일평균 분포 (500+) ===")
print(top_suspects['daily_avg'].describe())

- 20명 중 1명만 빼고 active_days가 0~1으로 확인됨
- 하루(또는 몇 시간) 동안 1,000번 이상 레슨에 진입했다

In [ ]:
sample_user = 'SAMPLE_USER_ID'

run_query(f"""
SELECT
    user_id,
    event_time,
    lesson_id,
    COUNT(*) AS dup_cnt
FROM v_events_lesson_view
WHERE user_id = '{sample_user}'
GROUP BY user_id, event_time, lesson_id
ORDER BY dup_cnt DESC
LIMIT 5;
""", "active0_check")

## Acquisition

### 뷰 테이블 생성

In [ ]:
create_bot_view_sql = """
DROP VIEW IF EXISTS v_bot_users;

CREATE VIEW v_bot_users AS
SELECT user_id
FROM v_events_lesson_view
GROUP BY user_id
HAVING COUNT(*) / GREATEST(
    TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)),
    1
) >= 200;
"""

execute_many(create_bot_view_sql)

In [ ]:
create_acquisition_views_sql = """
DROP VIEW IF EXISTS v_events_signup_clean;
CREATE VIEW v_events_signup_clean AS
SELECT
    user_id,
    CASE
        WHEN type IS NULL OR TRIM(type) = '' OR TRIM(type) = 'test' THEN 'unknown'
        ELSE type
    END AS signup_type,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time
FROM events_signup cs
WHERE NULLIF(TRIM(cs.user_id), '') IS NOT NULL
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM v_bot_users b
      WHERE b.user_id = cs.user_id
  );

DROP VIEW IF EXISTS v_events_content_start_clean;
CREATE VIEW v_events_content_start_clean AS
WITH signup AS (
    SELECT
        user_id,
        MIN(event_time) AS signup_time
    FROM v_events_signup_clean
    GROUP BY user_id
)
SELECT
    sc.user_id,
    STR_TO_DATE(sc.event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    sc.`content_id` AS content_id,
    sc.`content.difficulty` AS content_difficulty
FROM events_content_start sc
JOIN signup s
    ON sc.user_id = s.user_id
WHERE NULLIF(TRIM(sc.user_id), '') IS NOT NULL
  AND STR_TO_DATE(sc.event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL
  -- 정합성
  AND STR_TO_DATE(sc.event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') >= s.signup_time
  AND NOT EXISTS (
      SELECT 1
      FROM v_bot_users b
      WHERE b.user_id = sc.user_id
  );
"""

execute_many(create_acquisition_views_sql)

### 뷰 테이블 생성 검증

In [ ]:
run_query("""
SELECT 'v_bot_users' AS view_name, COUNT(*) AS row_cnt, COUNT(DISTINCT user_id) AS user_cnt
FROM v_bot_users
UNION ALL
SELECT 'v_events_signup_clean', COUNT(*), COUNT(DISTINCT user_id)
FROM v_events_signup_clean
UNION ALL
SELECT 'v_events_content_start_clean', COUNT(*), COUNT(DISTINCT user_id)
FROM v_events_content_start_clean;
""", "acquisition_view_check")


- 전처리 전후 비교
    - 회원가입 완료 유저 151명 감소
        - 145,133 -> 144,982
    - 콘텐츠 시작 유저 112명 감소
        - 38,652 -> 38,540

# 1. 유입 정의
*회원가입 + 콘텐츠 시작*

In [ ]:
query = """
SELECT
    COUNT(DISTINCT user_id) AS acquisition_users
FROM v_events_content_start_clean;
"""

pd.read_sql(query, engine)


## 유입 퍼널 시각화

In [ ]:
query = """
WITH funnel_counts AS (
    SELECT
        (SELECT COUNT(DISTINCT user_id) FROM v_events_signup_clean) AS signup_users,
        (SELECT COUNT(DISTINCT user_id) FROM v_events_content_start_clean) AS content_users
)

SELECT
    signup_users,
    content_users,
    ROUND(content_users / signup_users * 100, 1) AS convert_pct
FROM funnel_counts;
"""

ac_funnel_df = pd.read_sql(query, engine)
ac_funnel_df


In [ ]:
steps = ['회원가입 완료', '콘텐츠 시작']

counts = [
    ac_funnel_df['signup_users'].iloc[0],
    ac_funnel_df['content_users'].iloc[0]
]

rates = [
    None,
    ac_funnel_df['convert_pct'].iloc[0]
]

plt.figure(figsize=(10, 6))

colors = sns.color_palette('Blues', len(steps))
barplot = sns.barplot(x=steps, y=counts, palette=colors)

for i, count in enumerate(counts):
    plt.text(i, count+(max(counts)*0.02), f'{int(count):,}명',
    ha='center', va='bottom', fontsize=12, fontweight='bold')

    if i > 0 and rates[i] is not None:
        plt.text(i, count/2, f'({rates[i]}%)', ha='center', va='center',
        color='white', fontsize=11, fontweight='bold')

ax = plt.gca()
ax.spines[['top', 'left', 'right']].set_visible(False)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

plt.title('유입 퍼널', fontsize=15, fontweight='bold')
plt.ylabel('User Count')
plt.ylim(0, max(counts)*1.15)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 월별 유입 유저 수

In [ ]:
query = """
WITH content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
)

SELECT
    DATE_FORMAT(first_content_time, '%%Y년 %%m월') AS month,
    COUNT(DISTINCT user_id) AS acquisition_count
FROM content
WHERE first_content_time < '2024-01-01'
GROUP BY month
ORDER BY MIN(first_content_time)
"""

user_count_in_month = pd.read_sql(query, engine)
user_count_in_month


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

base_color = '#82A6CB'
highlight_color = '#F93441'

ax.plot(
    user_count_in_month['month'],
    user_count_in_month['acquisition_count'],
    color=base_color
)

max_sign = user_count_in_month['acquisition_count'].max()
max_month = user_count_in_month.loc[user_count_in_month['acquisition_count'].idxmax(), 'month']

ax.plot(max_month, max_sign, marker='o', markersize=5, color=highlight_color)
ax.annotate(
    f'Pick!\n{max_month}: {max_sign:,.0f}명',
    xy=(max_month, max_sign),
    xytext=(max_month, max_sign+150),
    ha='center', va='bottom', fontsize=10, color=highlight_color, fontweight='bold'
)

ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.set_title('월별 유입 유저 수', fontsize=15, fontweight='bold')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params('x', rotation=45)
ax.set_xlabel('유입 달')
ax.set_ylabel('유저 수(명)')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
ax.set_ylim(0, user_count_in_month['acquisition_count'].max()*1.2)

plt.tight_layout()
plt.show()

- 12월에 블프
- UTC+0
- 서비스 인지도 높아진 듯
- 2022년, 2023년 공통 특징으로는
    - 6월에서 급 상승
    - 8월 하락구간 시작
    - 10월 최저점 찍기
    - 12월 최고점 찍는 형태 <br>
    => 원인 파악 필요

# 월별 광고비

In [ ]:
query = """
WITH content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
),

user_by_month AS (
    SELECT
        DATE_FORMAT(first_content_time, '%%Y-%%m') AS month,
        COUNT(DISTINCT user_id) AS acquisition_count
    FROM content
    WHERE first_content_time < '2024-01-01'
    GROUP BY month
),

ad_by_month AS (
    SELECT
        DATE_FORMAT(spend_date, '%%Y-%%m') AS month,
        SUM(spend_krw) AS total_spend_krw
    FROM marketing_spend_daily
    WHERE spend_date < '2024-01-01'
    GROUP BY month
)

SELECT
    u.month,
    u.acquisition_count,
    a.total_spend_krw,
    ROUND(a.total_spend_krw / NULLIF(u.acquisition_count, 0), 0) AS cost_per_user
FROM user_by_month u
LEFT JOIN ad_by_month a
    ON u.month = a.month
ORDER BY u.month;
"""

cost_of_month = pd.read_sql(query, engine)
cost_of_month


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

base_color = '#82A6CB'
highlight_color = '#F93441'

ax.plot(
    cost_of_month['month'],
    cost_of_month['total_spend_krw'],
    color=base_color
)

max_sign = cost_of_month['total_spend_krw'].max()
max_month = cost_of_month.loc[cost_of_month['total_spend_krw'].idxmax(), 'month']

ax.plot(max_month, max_sign, marker='o', markersize=5, color=highlight_color)
ax.annotate(
    f'Pick!\n{max_month}: {max_sign:,.0f}원',
    xy=(max_month, max_sign),
    xytext=(max_month, max_sign*1.05),
    ha='center', va='bottom', fontsize=10, color=highlight_color, fontweight='bold'
)

ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.set_title('월별 광고 비용', fontsize=15, fontweight='bold')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params('x', rotation=45)
ax.set_xlabel('광고달')
ax.set_ylabel('광고비(원)')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
ax.set_ylim(0, cost_of_month['total_spend_krw'].max()*1.2)

plt.tight_layout()
plt.show()

# 세부 채널별 유입 비율

In [ ]:
query = """
WITH signup AS (
    SELECT
        user_id,
        MIN(event_time) AS signup_time
    FROM v_events_signup_clean
    GROUP BY user_id
),

content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
),

funnel_base AS (
    SELECT
        ua.user_id,
        ua.last_channel,
        s.signup_time,
        c.first_content_time
    FROM signup s
    INNER JOIN user_acquisition ua
        ON s.user_id = ua.user_id
    LEFT JOIN content c
        ON s.user_id = c.user_id
)

SELECT
    last_channel,
    COUNT(DISTINCT user_id) AS acquired_count,
    COUNT(DISTINCT user_id) AS signup_count,
    COUNT(DISTINCT CASE WHEN first_content_time IS NOT NULL THEN user_id END) AS content_start_count,
    ROUND(
        COUNT(DISTINCT CASE WHEN first_content_time IS NOT NULL THEN user_id END) * 100.0
        / NULLIF(COUNT(DISTINCT user_id), 0),
        1
    ) AS signup_to_content_pct
FROM funnel_base
GROUP BY last_channel
ORDER BY content_start_count DESC;
"""

channel_ac = pd.read_sql(query, engine)
channel_ac


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 10))

x = np.arange(len(channel_ac['last_channel']))
width = 0.35

bars1 = ax[0].bar(x - width/2, channel_ac['acquired_count'], width, label='회원가입 유저 (Signup)', color='#8ABED0')

for val, bar in zip(channel_ac['acquired_count'], bars1):
    ax[0].text(bar.get_x()+bar.get_width()/2, bar.get_height(),
               f'{val:,}명', ha='center', va='bottom', fontsize=8)

bars2 = ax[0].bar(x + width/2, channel_ac['content_start_count'], width, label='콘텐츠 시작 유저 (Started)', color='#4999B6')

for val, bar in zip(channel_ac['content_start_count'], bars2):
    ax[0].text(bar.get_x()+bar.get_width()/2, bar.get_height(),
               f'{val:,}명', ha='center', va='bottom', fontsize=8)

ax[0].set_ylabel('유저 수(명)')
ax[0].set_title('채널별 회원가입 및 콘텐츠 시작 유저 수', fontsize=15, fontweight='bold')
ax[0].set_xticks(x)
ax[0].set_xticklabels(channel_ac['last_channel'])
ax[0].spines[['top', 'left', 'right']].set_visible(False)
ax[0].grid(axis='y', linestyle='--', alpha=0.3)
ax[0].set_ylim(0, channel_ac['acquired_count'].max()*1.2)
ax[0].legend()



#--
channel_ac_sorted = channel_ac.sort_values(by='signup_to_content_pct', ascending=False).reset_index(drop=True)

x2 = np.arange(len(channel_ac_sorted['last_channel']))

bars = ax[1].bar(x2, channel_ac_sorted['signup_to_content_pct'], color='#EBCCD8', width=0.5)

ax[1].set_ylabel('전환율 (%)')
ax[1].set_title('채널별 콘텐츠 시작 전환율 (%)', fontsize=15, fontweight='bold')
ax[1].set_xticks(x2)
ax[1].set_xticklabels(channel_ac_sorted['last_channel'])
ax[1].spines[['top', 'left', 'right']].set_visible(False)
ax[1].set_ylim(0, channel_ac['signup_to_content_pct'].max()*1.2)
ax[1].grid(axis='y', linestyle='--', alpha=0.3)

for bar in bars:
    yval = bar.get_height()
    ax[1].text(bar.get_x()+bar.get_width()/2, yval+0.5, f'{yval}%',
               ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

- 일반 서치로 유입된 유저가 회원가입을 제일 많이 하나, 인원이 많은 만큼 전환율이 낮음.
- 서치 광고, 소셜 광고로 인해 서비스로 들어온 유저들이 다음으로 회원가입을 많이 함.
- 그러나 모든 전환율이 20%를 웃도는 양상.
    - 매력적인 콘텐츠를 이용한 개선 요구.

# 소스별 유입 비율

In [ ]:
query = """
WITH signup AS (
    SELECT
        user_id,
        MIN(event_time) AS signup_time
    FROM v_events_signup_clean
    GROUP BY user_id
),

content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
),

funnel_base AS (
    SELECT
        ua.user_id,
        ua.last_channel,
        ua.last_source,
        s.signup_time,
        c.first_content_time
    FROM signup s
    INNER JOIN user_acquisition ua
        ON s.user_id = ua.user_id
    LEFT JOIN content c
        ON s.user_id = c.user_id
)

SELECT
    last_channel,
    last_source,
    COUNT(DISTINCT user_id) AS signup_count,

    COUNT(DISTINCT CASE
        WHEN first_content_time IS NOT NULL
        THEN user_id
    END) AS content_start_count,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN first_content_time IS NOT NULL
            THEN user_id
        END) * 100.0
        / NULLIF(COUNT(DISTINCT user_id), 0),
        1
    ) AS signup_to_content_pct

FROM funnel_base
GROUP BY last_channel, last_source
ORDER BY signup_to_content_pct DESC;
"""

source_ac = pd.read_sql(query, engine)
source_ac


In [ ]:
plt.figure(figsize=(14, 8))

sns.scatterplot(
    data=source_ac,
    x='signup_count',
    y='signup_to_content_pct',
    hue='last_channel',
    s=150,
    alpha=0.8,
    palette='Set2'
)

texts = []
for i in range(len(source_ac)):
    t = plt.text(
        source_ac['signup_count'].iloc[i],
        source_ac['signup_to_content_pct'].iloc[i],
        source_ac['last_source'].iloc[i],
        fontsize=9,
        color='black'
    )
    texts.append(t)

adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.7))

plt.title('유입 소스별 회원가입 및 콘텐츠 시작 전환율', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('회원가입 유저 수 (명)', fontsize=12)
plt.ylabel('콘텐츠 시작 전환율 (%)', fontsize=12)

avg_signup = source_ac['signup_count'].mean()
avg_pct = source_ac['signup_to_content_pct'].mean()
plt.axvline(avg_signup, color='gray', linestyle='--', alpha=0.5, label='평균 회원가입 수')
plt.axhline(avg_pct, color='gray', linestyle='--', alpha=0.5, label='평균 전환율')

plt.legend(title='유입 채널 / 평균선', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# 마케팅 효율

In [ ]:
query = """
WITH signup AS (
    SELECT
        user_id,
        MIN(event_time) AS signup_time
    FROM v_events_signup_clean
    GROUP BY user_id
),

content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
),

channel_funnel AS (
    SELECT
        ua.last_channel AS channel,
        COUNT(DISTINCT s.user_id) AS signup_users,
        COUNT(DISTINCT c.user_id) AS content_users
    FROM signup s
    INNER JOIN user_acquisition ua
        ON s.user_id = ua.user_id
    LEFT JOIN content c
        ON s.user_id = c.user_id
    GROUP BY ua.last_channel
),

ad_metrics AS (
    SELECT
        channel,
        SUM(spend_krw) AS total_spend,
        SUM(clicks) AS total_clicks
    FROM marketing_spend_daily
    GROUP BY channel
)

SELECT
    am.channel,
    am.total_spend,
    am.total_clicks,
    cf.signup_users,
    cf.content_users,

    CASE
        WHEN cf.signup_users > 0
        THEN ROUND(am.total_spend / cf.signup_users, 0)
        ELSE 0
    END AS signup_CAC,

    CASE
        WHEN cf.content_users > 0
        THEN ROUND(am.total_spend / cf.content_users, 0)
        ELSE 0
    END AS content_CAC,

    CASE
        WHEN am.total_clicks > 0
        THEN ROUND(cf.signup_users * 100.0 / am.total_clicks, 2)
        ELSE 0
    END AS click_to_signup_rate,

    CASE
        WHEN am.total_clicks > 0
        THEN ROUND(cf.content_users * 100.0 / am.total_clicks, 2)
        ELSE 0
    END AS click_to_content_rate

FROM ad_metrics am
INNER JOIN channel_funnel cf
    ON am.channel = cf.channel
ORDER BY content_CAC;
"""

marketing_efficiency = pd.read_sql(query, engine)
marketing_efficiency


# 연령대별 유입 비율

In [ ]:
query = """
WITH signup AS (
    SELECT
        user_id,
        MIN(event_time) AS signup_time
    FROM v_events_signup_clean
    GROUP BY user_id
),

content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
),

user_funnel AS (
    SELECT
        up.age_group,

        COUNT(DISTINCT s.user_id) AS signup_users,

        COUNT(DISTINCT CASE
            WHEN c.first_content_time IS NOT NULL
            THEN c.user_id
        END) AS content_users

    FROM signup s

    INNER JOIN user_dim up
        ON s.user_id = up.user_id

    LEFT JOIN content c
        ON s.user_id = c.user_id

    GROUP BY up.age_group
)

SELECT
    age_group,

    signup_users AS total_signup,

    ROUND(
        signup_users * 100.0
        / SUM(signup_users) OVER (),
        1
    ) AS signup_pct,

    content_users AS total_events_content_start,

    ROUND(
        content_users * 100.0
        / SUM(content_users) OVER (),
        1
    ) AS content_pct,

    ROUND(
        content_users * 100.0
        / NULLIF(signup_users, 0),
        1
    ) AS acquire_pct

FROM user_funnel
ORDER BY signup_pct DESC
"""

age_acquisition = pd.read_sql(query, engine)
age_acquisition

# 직업별 유입 비율

In [ ]:
query = """
WITH signup AS (
    SELECT
        user_id,
        MIN(event_time) AS signup_time
    FROM v_events_signup_clean
    GROUP BY user_id
),

content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
),

user_funnel AS (
    SELECT
        up.occupation,

        COUNT(DISTINCT s.user_id) AS signup_users,

        COUNT(DISTINCT CASE
            WHEN c.first_content_time IS NOT NULL
            THEN c.user_id
        END) AS content_users

    FROM signup s
    INNER JOIN user_dim up
        ON s.user_id = up.user_id
    LEFT JOIN content c
        ON s.user_id = c.user_id
    GROUP BY up.occupation
)

SELECT
    occupation,

    signup_users AS total_signup,

    ROUND(signup_users * 100.0 / SUM(signup_users) OVER (), 1) AS signup_pct,

    content_users AS total_events_content_start,

    ROUND(content_users * 100.0 / SUM(content_users) OVER (), 1) AS content_pct,

    ROUND(content_users * 100.0 / NULLIF(signup_users, 0), 1) AS acquire_pct

FROM user_funnel
ORDER BY total_events_content_start DESC;
"""

occupation_acquisition = pd.read_sql(query, engine)
occupation_acquisition

# 관심 카테고리별 유입 비율

In [ ]:
query = """
WITH signup AS (
    SELECT
        user_id,
        MIN(event_time) AS signup_time
    FROM v_events_signup_clean
    GROUP BY user_id
),

content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
),

user_funnel AS (
    SELECT
        up.primary_interest,

        COUNT(DISTINCT s.user_id) AS signup_users,

        COUNT(DISTINCT CASE
            WHEN c.first_content_time IS NOT NULL
            THEN c.user_id
        END) AS content_users

    FROM signup s

    INNER JOIN user_dim up
        ON s.user_id = up.user_id

    LEFT JOIN content c
        ON s.user_id = c.user_id

    GROUP BY up.primary_interest
)

SELECT
    primary_interest,

    signup_users AS total_signup,

    ROUND(
        signup_users * 100.0
        / SUM(signup_users) OVER (),
        1
    ) AS signup_pct,

    content_users AS total_events_content_start,

    ROUND(
        content_users * 100.0
        / SUM(content_users) OVER (),
        1
    ) AS content_pct,

    ROUND(
        content_users * 100.0
        / NULLIF(signup_users, 0),
        1
    ) AS acquire_pct

FROM user_funnel
ORDER BY total_events_content_start DESC;
"""

interest_acquisition = pd.read_sql(query, engine)
interest_acquisition

# 회원가입 경로별 유입 유저 비율

In [ ]:
query = """
WITH signup AS (
    SELECT
        user_id,
        signup_type,
        MIN(event_time) AS signup_time
    FROM v_events_signup_clean
    GROUP BY user_id, signup_type
),

content AS (
    SELECT
        user_id,
        MIN(event_time) AS first_content_time
    FROM v_events_content_start_clean
    GROUP BY user_id
),

user_funnel AS (
    SELECT
        s.signup_type,
        COUNT(DISTINCT s.user_id) AS signup_users,
        COUNT(DISTINCT c.user_id) AS content_users
    FROM signup s
    LEFT JOIN content c
        ON s.user_id = c.user_id
    GROUP BY s.signup_type
)

SELECT
    signup_type,
    signup_users AS total_signup,
    ROUND(signup_users * 100.0 / SUM(signup_users) OVER (), 1) AS signup_pct,

    content_users AS total_events_content_start,
    ROUND(content_users * 100.0 / SUM(content_users) OVER (), 1) AS content_pct,

    ROUND(content_users * 100.0 / NULLIF(signup_users, 0), 1) AS acquire_pct
FROM user_funnel
ORDER BY total_events_content_start DESC;
"""

signup_type_acquisition = pd.read_sql(query, engine)
signup_type_acquisition

# 콘텐츠 난이도별 유입 유저 비율

In [ ]:
query = """
WITH first_content AS (
    SELECT
        user_id,
        content_id,
        event_time AS first_content_time,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS rn
    FROM v_events_content_start_clean
),

funnel_base AS (
    SELECT
        fc.user_id,
        cm.difficulty
    FROM first_content fc
    LEFT JOIN content_dim cm
        ON fc.content_id = cm.content_id
    WHERE fc.rn = 1
),

total AS (
    SELECT
        (SELECT COUNT(DISTINCT user_id) FROM v_events_signup_clean) AS signup_users,
        (SELECT COUNT(DISTINCT user_id) FROM funnel_base) AS content_users
)

SELECT
    fb.difficulty,
    COUNT(DISTINCT fb.user_id) AS content_users,

    ROUND(
        COUNT(DISTINCT fb.user_id) * 100.0
        / NULLIF((SELECT signup_users FROM total), 0),
        1
    ) AS convert_content_pct,

    ROUND(
        COUNT(DISTINCT fb.user_id) * 100.0
        / NULLIF((SELECT content_users FROM total), 0),
        1
    ) AS content_start_pct

FROM funnel_base fb
WHERE fb.difficulty IS NOT NULL
GROUP BY fb.difficulty
ORDER BY content_start_pct DESC;
"""

difficulty_acquisition = pd.read_sql(query, engine)
difficulty_acquisition

# 콘텐츠 카테고리별 유입 유저 비율

In [ ]:
query = """
WITH first_content AS (
    SELECT
        user_id,
        content_id,
        event_time AS first_content_time,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS rn
    FROM v_events_content_start_clean
),

funnel_base AS (
    SELECT
        fc.user_id,
        cm.category
    FROM first_content fc
    LEFT JOIN content_dim cm
        ON fc.content_id = cm.content_id
    WHERE fc.rn = 1
),

total AS (
    SELECT
        (SELECT COUNT(DISTINCT user_id) FROM v_events_signup_clean) AS signup_users,
        (SELECT COUNT(DISTINCT user_id) FROM funnel_base) AS content_users
)

SELECT
    fb.category,
    COUNT(DISTINCT fb.user_id) AS content_users,

    ROUND(
        COUNT(DISTINCT fb.user_id) * 100.0
        / NULLIF((SELECT signup_users FROM total), 0),
        1
    ) AS convert_content_pct,

    ROUND(
        COUNT(DISTINCT fb.user_id) * 100.0
        / NULLIF((SELECT content_users FROM total), 0),
        1
    ) AS content_start_pct

FROM funnel_base fb
WHERE fb.category IS NOT NULL
GROUP BY fb.category
ORDER BY content_start_pct DESC;
"""

category_acquisition = pd.read_sql(query, engine)
category_acquisition


# 카테고리-난이도별 콘텐츠 개수

In [ ]:
query = '''
SELECT
    category,
    difficulty,
    COUNT(DISTINCT content_id) AS total_count
FROM content_dim
WHERE difficulty IS NOT NULL
GROUP BY category, difficulty
ORDER BY total_count DESC
'''

pd.read_sql(query, engine)